In [1]:
import os
import pandas as pd
import pickle as pkl
import logging
from tabulate import tabulate
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import roc_auc_score, accuracy_score, classification_report
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import StandardScaler
from pathlib import Path
import joblib
import sys
from rdkit.Chem import AllChem
import numpy as np

from featurise import FeatureExtractor
from train_model import ModelTraining  # Import your ModelTraining class
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"


import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)


/Users/amarachiordor/Documents/Outreachy/outreachy-contributions/scripts/featurise.py:15: FutureWarning: In the future `np.bool` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, 'bool'):


In [2]:
featurizer_id = "eos8a4x"
    
# Initialize the FeatureExtractor object
extractor = FeatureExtractor(featurizer_id)

# Example SMILES string (Ethanol)
smiles_string = "CCO"

# Call the featurize_smiles method to process the single SMILES string
features_df = extractor.featurize_smiles(smiles_string)

if features_df is not None:
    print("Featurization successful! Features extracted:")
    print(features_df)
else:
    print("Featurization failed.")

2025-04-09 09:01:34,964 - INFO - Featurizing SMILES string with model 'eos8a4x'...
INFO:featurise:Featurizing SMILES string with model 'eos8a4x'...
2025-04-09 09:01:37,882 - INFO - ⬇️  Fetching model eos8a4x: rdkit-descriptors
👎 Model eos8a4x failed to fetch! Model already exists on your system. If you want to fetch it again, please delete the existing model first.

INFO:featurise:⬇️  Fetching model eos8a4x: rdkit-descriptors
👎 Model eos8a4x failed to fetch! Model already exists on your system. If you want to fetch it again, please delete the existing model first.

2025-04-09 09:01:37,884 - INFO - Serving the model in the background...
INFO:featurise:Serving the model in the background...
2025-04-09 09:01:55,838 - INFO - 🚀 Serving model eos8a4x: rdkit-descriptors

   URL: http://0.0.0.0:54460
   PID: -1
   SRV: pulled_docker
   Output source: local-only

👉 To run model:
   - run

💁 Information:
   - info

INFO:featurise:🚀 Serving model eos8a4x: rdkit-descriptors

   URL: http://0.0.0.0

Featurization successful! Features extracted:
   BalabanJ   BertzCT      Chi0    Chi0n    Chi0v      Chi1     Chi1n  \
0  1.632993  2.754888  2.707107  2.15432  2.15432  1.414214  1.023335   

      Chi1v     Chi2n     Chi2v  ...  fr_sulfonamd  fr_sulfone  \
0  1.023335  0.316228  0.316228  ...           0.0         0.0   

   fr_term_acetylene  fr_tetrazole  fr_thiazole  fr_thiocyan  fr_thiophene  \
0                0.0           0.0          0.0          0.0           0.0   

   fr_unbrch_alkane  fr_urea       qed  
0               0.0      0.0  0.406808  

[1 rows x 200 columns]


In [3]:
extractor = FeatureExtractor(featurizer_id)
extractor.generate_features()

2025-04-09 09:02:19,420 - INFO - Fetching model 'eos8a4x' from Ersilia...
INFO:featurise:Fetching model 'eos8a4x' from Ersilia...
2025-04-09 09:02:21,208 - INFO - ⬇️  Fetching model eos8a4x: rdkit-descriptors
👎 Model eos8a4x failed to fetch! Model already exists on your system. If you want to fetch it again, please delete the existing model first.

INFO:featurise:⬇️  Fetching model eos8a4x: rdkit-descriptors
👎 Model eos8a4x failed to fetch! Model already exists on your system. If you want to fetch it again, please delete the existing model first.

2025-04-09 09:02:21,208 - INFO - Serving the model in the background...
INFO:featurise:Serving the model in the background...
2025-04-09 09:02:37,985 - INFO - 🚀 Serving model eos8a4x: rdkit-descriptors

   URL: http://0.0.0.0:54485
   PID: -1
   SRV: pulled_docker
   Output source: local-only

👉 To run model:
   - run

💁 Information:
   - info

INFO:featurise:🚀 Serving model eos8a4x: rdkit-descriptors

   URL: http://0.0.0.0:54485
   PID: -1


In [4]:
featurizer_id = "eos8a4x"  # Change if running training
model_type = "random_forest"
cv_strategy = "stratified_kfold"

# Initialize the ModelTraining class (training part)
trainer = ModelTraining(
    featurizer_id=featurizer_id,
    model_type=model_type,
    cv_strategy=cv_strategy
)
trainer.run()

# Generate filenames using all 3 parameters
filenames = trainer.generate_filenames(featurizer_id, model_type, cv_strategy)

print("\n📁 Model Artifacts:")
for key, filename in filenames.items():
    print(f"{key.capitalize()} File: {filename}")

INFO:train_model:Removed 8 zero-variance columns.
INFO:train_model:Feature columns saved to: /Users/amarachiordor/Documents/Outreachy/outreachy-contributions/models/eos8a4x_random_forest_stratified_kfold_feature_columns.txt
INFO:train_model:Zero-variance columns saved to: /Users/amarachiordor/Documents/Outreachy/outreachy-contributions/models/eos8a4x_random_forest_stratified_kfold_zero_variance_columns.txt
INFO:train_model:✅ Scaler saved to: /Users/amarachiordor/Documents/Outreachy/outreachy-contributions/models/eos8a4x_random_forest_stratified_kfold_scaler.pkl
INFO:train_model:SMOTE resampling completed.
INFO:train_model:Random Oversampling completed.
INFO:train_model:SMOTE + ENN Hybrid Resampling completed.
INFO:train_model:🔁 Running manual StratifiedKFold for RANDOM_FOREST...


Best Class 0 Precision (manual CV): 0.9428

Train Metrics:
╒══════════════════════╤═════════╕
│ Metric               │   Score │
╞══════════════════════╪═════════╡
│ Accuracy             │  0.9831 │
├──────────────────────┼─────────┤
│ Precision (Weighted) │  0.9832 │
├──────────────────────┼─────────┤
│ Recall (Weighted)    │  0.9831 │
├──────────────────────┼─────────┤
│ F1 Score (Weighted)  │  0.9831 │
├──────────────────────┼─────────┤
│ ROC AUC              │  0.9991 │
╘══════════════════════╧═════════╛

Classification Report:

              precision    recall  f1-score   support

           0       0.96      0.97      0.97       365
           1       0.99      0.99      0.99      1056

    accuracy                           0.98      1421
   macro avg       0.98      0.98      0.98      1421
weighted avg       0.98      0.98      0.98      1421


Validation Metrics:
╒══════════════════════╤═════════╕
│ Metric               │   Score │
╞══════════════════════╪═════════╡
│ Accura

In [5]:
smiles = "CC(C)CCOCC"  # Example SMILES string
featuriser = 'eos8a4x'
model_type = "random_forest"  # Specify model type
cv_strategy = "stratified_kfold"  # Specify the cross-validation strategy

# Use the FeatureExtractor to get features from the SMILES string
featurizer = FeatureExtractor(featuriser)
X = featurizer.featurize_smiles(smiles)

if X is not None:
    # Initialize ModelTraining for prediction, passing model_type and cv_strategy
    modelling = ModelTraining(featurizer_id=featuriser, model_type=model_type, cv_strategy=cv_strategy)

    # Now use the already trained scaler and model
    prediction = modelling.make_predictions(X, featuriser, model_type)  # Pass model_type



2025-04-09 09:04:41,753 - INFO - Featurizing SMILES string with model 'eos8a4x'...
INFO:featurise:Featurizing SMILES string with model 'eos8a4x'...
2025-04-09 09:04:44,460 - INFO - ⬇️  Fetching model eos8a4x: rdkit-descriptors
👎 Model eos8a4x failed to fetch! Model already exists on your system. If you want to fetch it again, please delete the existing model first.

INFO:featurise:⬇️  Fetching model eos8a4x: rdkit-descriptors
👎 Model eos8a4x failed to fetch! Model already exists on your system. If you want to fetch it again, please delete the existing model first.

2025-04-09 09:04:44,461 - INFO - Serving the model in the background...
INFO:featurise:Serving the model in the background...
2025-04-09 09:05:01,407 - INFO - 🚀 Serving model eos8a4x: rdkit-descriptors

   URL: http://0.0.0.0:54541
   PID: -1
   SRV: pulled_docker
   Output source: local-only

👉 To run model:
   - run

💁 Information:
   - info

INFO:featurise:🚀 Serving model eos8a4x: rdkit-descriptors

   URL: http://0.0.0.0


🧪 Prediction Result:
╒═════════════╤═══════════╕
│ Metric      │ Value     │
╞═════════════╪═══════════╡
│ Prediction  │ Permeable │
├─────────────┼───────────┤
│ Probability │ 0.8094    │
╘═════════════╧═══════════╛


## Using a Different Featuriser

In [6]:
featurizer_id = "eos5axz"
    
# Initialize the FeatureExtractor object
extractor = FeatureExtractor(featurizer_id)

# Example SMILES string (Ethanol)
smiles_string = "CCO"

# Call the featurize_smiles method to process the single SMILES string
features_df = extractor.featurize_smiles(smiles_string)

if features_df is not None:
    print("Featurization successful! Features extracted:")
    print(features_df)
else:
    print("Featurization failed.")

2025-04-09 09:05:25,768 - INFO - Featurizing SMILES string with model 'eos5axz'...
INFO:featurise:Featurizing SMILES string with model 'eos5axz'...
2025-04-09 09:05:27,739 - INFO - ⬇️  Fetching model eos5axz: morgan-counts
👎 Model eos5axz failed to fetch! Model already exists on your system. If you want to fetch it again, please delete the existing model first.

INFO:featurise:⬇️  Fetching model eos5axz: morgan-counts
👎 Model eos5axz failed to fetch! Model already exists on your system. If you want to fetch it again, please delete the existing model first.

2025-04-09 09:05:27,741 - INFO - Serving the model in the background...
INFO:featurise:Serving the model in the background...
2025-04-09 09:05:32,866 - INFO - 🚀 Serving model eos5axz: morgan-counts

   URL: http://0.0.0.0:54564
   PID: 66568
   SRV: conda
   Output source: local-only

👉 To run model:
   - run

💁 Information:
   - info

INFO:featurise:🚀 Serving model eos5axz: morgan-counts

   URL: http://0.0.0.0:54564
   PID: 66568


Featurization successful! Features extracted:
   dim_0000  dim_0001  dim_0002  dim_0003  dim_0004  dim_0005  dim_0006  \
0         0         0         0         0         0         0         0   

   dim_0007  dim_0008  dim_0009  ...  dim_2038  dim_2039  dim_2040  dim_2041  \
0         0         0         0  ...         0         0         0         0   

   dim_2042  dim_2043  dim_2044  dim_2045  dim_2046  dim_2047  
0         0         0         0         0         0         0  

[1 rows x 2048 columns]


In [7]:
extractor = FeatureExtractor(featurizer_id)
extractor.generate_features()

2025-04-09 09:05:56,737 - INFO - Fetching model 'eos5axz' from Ersilia...
INFO:featurise:Fetching model 'eos5axz' from Ersilia...
2025-04-09 09:05:58,589 - INFO - ⬇️  Fetching model eos5axz: morgan-counts
👎 Model eos5axz failed to fetch! Model already exists on your system. If you want to fetch it again, please delete the existing model first.

INFO:featurise:⬇️  Fetching model eos5axz: morgan-counts
👎 Model eos5axz failed to fetch! Model already exists on your system. If you want to fetch it again, please delete the existing model first.

2025-04-09 09:05:58,591 - INFO - Serving the model in the background...
INFO:featurise:Serving the model in the background...
2025-04-09 09:06:03,317 - INFO - 🚀 Serving model eos5axz: morgan-counts

   URL: http://0.0.0.0:54597
   PID: 66751
   SRV: conda
   Output source: local-only

👉 To run model:
   - run

💁 Information:
   - info

INFO:featurise:🚀 Serving model eos5axz: morgan-counts

   URL: http://0.0.0.0:54597
   PID: 66751
   SRV: conda
   O

In [8]:
featurizer_id = "eos5axz"  # Change if running training
model_type = "random_forest"
cv_strategy = "stratified_kfold"

# Initialize the ModelTraining class (training part)
trainer = ModelTraining(
    featurizer_id=featurizer_id,
    model_type=model_type,
    cv_strategy=cv_strategy
)
trainer.run()

# Generate filenames using all 3 parameters
filenames = trainer.generate_filenames(featurizer_id, model_type, cv_strategy)

print("\n📁 Model Artifacts:")
for key, filename in filenames.items():
    print(f"{key.capitalize()} File: {filename}")

INFO:train_model:Removed 0 zero-variance columns.
INFO:train_model:Feature columns saved to: /Users/amarachiordor/Documents/Outreachy/outreachy-contributions/models/eos5axz_random_forest_stratified_kfold_feature_columns.txt
INFO:train_model:Zero-variance columns saved to: /Users/amarachiordor/Documents/Outreachy/outreachy-contributions/models/eos5axz_random_forest_stratified_kfold_zero_variance_columns.txt
INFO:train_model:✅ Scaler saved to: /Users/amarachiordor/Documents/Outreachy/outreachy-contributions/models/eos5axz_random_forest_stratified_kfold_scaler.pkl
INFO:train_model:SMOTE resampling completed.
INFO:train_model:Random Oversampling completed.
INFO:train_model:SMOTE + ENN Hybrid Resampling completed.
INFO:train_model:🔁 Running manual StratifiedKFold for RANDOM_FOREST...


Best Class 0 Precision (manual CV): 0.9734

Train Metrics:
╒══════════════════════╤═════════╕
│ Metric               │   Score │
╞══════════════════════╪═════════╡
│ Accuracy             │  0.931  │
├──────────────────────┼─────────┤
│ Precision (Weighted) │  0.9334 │
├──────────────────────┼─────────┤
│ Recall (Weighted)    │  0.931  │
├──────────────────────┼─────────┤
│ F1 Score (Weighted)  │  0.9281 │
├──────────────────────┼─────────┤
│ ROC AUC              │  0.9829 │
╘══════════════════════╧═════════╛

Classification Report:

              precision    recall  f1-score   support

           0       0.97      0.76      0.85       365
           1       0.92      0.99      0.96      1056

    accuracy                           0.93      1421
   macro avg       0.94      0.87      0.90      1421
weighted avg       0.93      0.93      0.93      1421


Validation Metrics:
╒══════════════════════╤═════════╕
│ Metric               │   Score │
╞══════════════════════╪═════════╡
│ Accura

In [9]:
smiles = "CC(C)CCOCC"  # Example SMILES string
featuriser = 'eos5axz'
model_type = "random_forest"  # Specify model type
cv_strategy = "stratified_kfold"  # Specify the cross-validation strategy

# Use the FeatureExtractor to get features from the SMILES string
featurizer = FeatureExtractor(featuriser)
X = featurizer.featurize_smiles(smiles)

if X is not None:
    # Initialize ModelTraining for prediction, passing model_type and cv_strategy
    modelling = ModelTraining(featurizer_id=featuriser, model_type=model_type, cv_strategy=cv_strategy)

    # Now use the already trained scaler and model
    prediction = modelling.make_predictions(X, featuriser, model_type)  # Pass model_type

2025-04-09 09:07:37,195 - INFO - Featurizing SMILES string with model 'eos5axz'...
INFO:featurise:Featurizing SMILES string with model 'eos5axz'...
2025-04-09 09:07:39,859 - INFO - ⬇️  Fetching model eos5axz: morgan-counts
👎 Model eos5axz failed to fetch! Model already exists on your system. If you want to fetch it again, please delete the existing model first.

INFO:featurise:⬇️  Fetching model eos5axz: morgan-counts
👎 Model eos5axz failed to fetch! Model already exists on your system. If you want to fetch it again, please delete the existing model first.

2025-04-09 09:07:39,860 - INFO - Serving the model in the background...
INFO:featurise:Serving the model in the background...
2025-04-09 09:07:44,783 - INFO - 🚀 Serving model eos5axz: morgan-counts

   URL: http://0.0.0.0:54670
   PID: 67312
   SRV: conda
   Output source: local-only

👉 To run model:
   - run

💁 Information:
   - info

INFO:featurise:🚀 Serving model eos5axz: morgan-counts

   URL: http://0.0.0.0:54670
   PID: 67312



🧪 Prediction Result:
╒═════════════╤═══════════╕
│ Metric      │ Value     │
╞═════════════╪═══════════╡
│ Prediction  │ Permeable │
├─────────────┼───────────┤
│ Probability │ 0.7063    │
╘═════════════╧═══════════╛


## Using Different ML Architecture

In [10]:
featurizer_id = "eos5axz"  # Change if running training
model_type = "xgboost"
cv_strategy = "grid_search"


# Initialize the ModelTraining class (training part)
trainer = ModelTraining(
    featurizer_id=featurizer_id,
    model_type=model_type,
    cv_strategy=cv_strategy
)
trainer.run()

# Generate filenames using all 3 parameters
filenames = trainer.generate_filenames(featurizer_id, model_type, cv_strategy)

print("\n📁 Model Artifacts:")
for key, filename in filenames.items():
    print(f"{key.capitalize()} File: {filename}")

INFO:train_model:Removed 0 zero-variance columns.
INFO:train_model:Feature columns saved to: /Users/amarachiordor/Documents/Outreachy/outreachy-contributions/models/eos5axz_xgboost_grid_search_feature_columns.txt
INFO:train_model:Zero-variance columns saved to: /Users/amarachiordor/Documents/Outreachy/outreachy-contributions/models/eos5axz_xgboost_grid_search_zero_variance_columns.txt
INFO:train_model:✅ Scaler saved to: /Users/amarachiordor/Documents/Outreachy/outreachy-contributions/models/eos5axz_xgboost_grid_search_scaler.pkl
INFO:train_model:SMOTE resampling completed.
INFO:train_model:Random Oversampling completed.
INFO:train_model:SMOTE + ENN Hybrid Resampling completed.
INFO:train_model:🔍 Running GridSearchCV for XGBOOST...


Fitting 5 folds for each of 8 candidates, totalling 40 fits
Best Hyperparameters: {'colsample_bytree': 0.8, 'learning_rate': 0.1, 'max_depth': 6, 'n_estimators': 100, 'subsample': 0.8}
Best Class 0 Precision (CV): 0.8911

Train Metrics:
╒══════════════════════╤═════════╕
│ Metric               │   Score │
╞══════════════════════╪═════════╡
│ Accuracy             │  0.969  │
├──────────────────────┼─────────┤
│ Precision (Weighted) │  0.9689 │
├──────────────────────┼─────────┤
│ Recall (Weighted)    │  0.969  │
├──────────────────────┼─────────┤
│ F1 Score (Weighted)  │  0.9688 │
├──────────────────────┼─────────┤
│ ROC AUC              │  0.997  │
╘══════════════════════╧═════════╛

Classification Report:

              precision    recall  f1-score   support

           0       0.96      0.92      0.94       365
           1       0.97      0.99      0.98      1056

    accuracy                           0.97      1421
   macro avg       0.97      0.95      0.96      1421
weighted av

In [11]:
featurizer_id = "eos8a4x"  # Change if running training
model_type = "xgboost"
cv_strategy = "grid_search"


# Initialize the ModelTraining class (training part)
trainer = ModelTraining(
    featurizer_id=featurizer_id,
    model_type=model_type,
    cv_strategy=cv_strategy
)
trainer.run()

# Generate filenames using all 3 parameters
filenames = trainer.generate_filenames(featurizer_id, model_type, cv_strategy)

print("\n📁 Model Artifacts:")
for key, filename in filenames.items():
    print(f"{key.capitalize()} File: {filename}")

INFO:train_model:Removed 8 zero-variance columns.
INFO:train_model:Feature columns saved to: /Users/amarachiordor/Documents/Outreachy/outreachy-contributions/models/eos8a4x_xgboost_grid_search_feature_columns.txt
INFO:train_model:Zero-variance columns saved to: /Users/amarachiordor/Documents/Outreachy/outreachy-contributions/models/eos8a4x_xgboost_grid_search_zero_variance_columns.txt
INFO:train_model:✅ Scaler saved to: /Users/amarachiordor/Documents/Outreachy/outreachy-contributions/models/eos8a4x_xgboost_grid_search_scaler.pkl
INFO:train_model:SMOTE resampling completed.
INFO:train_model:Random Oversampling completed.
INFO:train_model:SMOTE + ENN Hybrid Resampling completed.
INFO:train_model:🔍 Running GridSearchCV for XGBOOST...


Fitting 5 folds for each of 8 candidates, totalling 40 fits
Best Hyperparameters: {'colsample_bytree': 0.8, 'learning_rate': 0.1, 'max_depth': 6, 'n_estimators': 200, 'subsample': 0.8}
Best Class 0 Precision (CV): 0.8926

Train Metrics:
╒══════════════════════╤═════════╕
│ Metric               │   Score │
╞══════════════════════╪═════════╡
│ Accuracy             │  0.993  │
├──────────────────────┼─────────┤
│ Precision (Weighted) │  0.993  │
├──────────────────────┼─────────┤
│ Recall (Weighted)    │  0.993  │
├──────────────────────┼─────────┤
│ F1 Score (Weighted)  │  0.993  │
├──────────────────────┼─────────┤
│ ROC AUC              │  0.9999 │
╘══════════════════════╧═════════╛

Classification Report:

              precision    recall  f1-score   support

           0       0.98      0.99      0.99       365
           1       1.00      0.99      1.00      1056

    accuracy                           0.99      1421
   macro avg       0.99      0.99      0.99      1421
weighted av